# Ejercicio: emociones faciales con cámara en Google Colab

El usuario autoriza la cámara, toma una captura y el modelo estima una emoción predominante. No identifica personas ni guarda video.

> Use esta práctica con consentimiento. Las emociones son estimaciones probabilísticas y no deben usarse para evaluar personas, tomar decisiones laborales o inferir estados psicológicos reales.

## 1. Instalar dependencias

DeepFace es una biblioteca que integra modelos preentrenados para análisis facial. En este ejercicio solamente activaremos la acción `emotion`; no se utilizará reconocimiento de identidad. La primera ejecución puede tardar algunos minutos porque Colab descarga las bibliotecas y, al analizar la primera imagen, los pesos del modelo.

**Qué debe observar:** al finalizar debe aparecer `Dependencias listas`. Si aparece un error de instalación, reinicie el entorno de ejecución y ejecute esta celda nuevamente.

In [ ]:
!pip -q install deepface==0.0.100

import deepface
from deepface import DeepFace
import base64
import cv2
import matplotlib.pyplot as plt
import numpy as np
from google.colab import output
from IPython.display import Javascript, display

print('Dependencias listas')
print('Versión de DeepFace:', deepface.__version__)

## 2. Abrir la cámara y tomar una captura

Este bloque utiliza JavaScript del navegador porque el entorno remoto de Colab no puede abrir directamente la cámara del equipo. El navegador solicitará permiso; la cámara se activa durante unos segundos, se toma una sola imagen y la transmisión se detiene.

**Qué debe observar:** debe aparecer la solicitud de permiso de cámara. Al terminar, verá la ruta temporal `/content/captura_emocion.jpg`. La imagen no se descarga ni se guarda en una base de datos; permanece únicamente en la sesión temporal de Colab.

In [ ]:
display(Javascript(r'''
async function captureEmotionFrame() {
  const video = document.createElement('video');
  video.width = 640;
  video.height = 480;
  video.autoplay = true;
  video.style.border = '3px solid #1f77b4';
  document.body.appendChild(video);
  const stream = await navigator.mediaDevices.getUserMedia({video: true, audio: false});
  video.srcObject = stream;
  await new Promise(resolve => setTimeout(resolve, 2500));
  const canvas = document.createElement('canvas');
  canvas.width = video.videoWidth || 640;
  canvas.height = video.videoHeight || 480;
  canvas.getContext('2d').drawImage(video, 0, 0, canvas.width, canvas.height);
  stream.getTracks().forEach(track => track.stop());
  video.remove();
  return canvas.toDataURL('image/jpeg', 0.85);
}
'''))

image_data = output.eval_js('captureEmotionFrame()')
image_bytes = base64.b64decode(image_data.split(',')[1])
image_path = '/content/captura_emocion.jpg'
with open(image_path, 'wb') as image_file:
    image_file.write(image_bytes)

print('Captura guardada temporalmente en:', image_path)

## 3. Analizar la emoción estimada

DeepFace analiza la captura y devuelve una puntuación para cada categoría: `happy` (feliz), `sad` (triste), `angry` (enojo), `surprise` (sorpresa), `fear` (miedo), `disgust` (disgusto) y `neutral` (neutral).

`dominant_emotion` es la categoría con mayor puntuación; no significa que el modelo tenga certeza absoluta. Por ejemplo, una salida de `happy: 62%` significa que, entre las categorías del modelo, esa fue la hipótesis con mayor puntuación para esa imagen. No debe interpretarse como una medición clínica ni como una prueba de cómo se siente realmente la persona.

`enforce_detection=False` permite que el ejercicio continúe aunque el detector no encuentre claramente un rostro. En ese caso, las puntuaciones pueden ser poco confiables; por eso conviene repetir la captura con buena iluminación, el rostro de frente y sin obstrucciones.

In [ ]:
analysis = DeepFace.analyze(
    img_path=image_path,
    actions=['emotion'],
    enforce_detection=False,
    detector_backend='opencv',
    silent=True
)
if isinstance(analysis, list):
    analysis = analysis[0]

dominant_emotion = analysis.get('dominant_emotion', 'no disponible')
image = cv2.cvtColor(cv2.imread(image_path), cv2.COLOR_BGR2RGB)
plt.figure(figsize=(8, 5))
plt.imshow(image)
plt.axis('off')
plt.title(f'Emoción estimada: {dominant_emotion}')
plt.show()

print('Emoción predominante:', dominant_emotion)
print('Puntuaciones:')
for emotion, score in sorted(analysis['emotion'].items(), key=lambda item: item[1], reverse=True):
    print(f'  {emotion:10s}: {score:6.2f}%')

## 4. Interpretar y repetir resultados

Para tomar otra captura, vuelva a ejecutar la celda 2 y luego la celda 3. Compare las puntuaciones, no solamente la emoción predominante. Una diferencia pequeña entre las dos primeras categorías indica que el resultado es ambiguo; una puntuación alta y claramente separada indica mayor consistencia del modelo para esa imagen, pero no garantiza exactitud.

### Interpretación recomendada

- **Resultado esperado:** una imagen, una emoción predominante y una lista de porcentajes.
- **Sin rostro detectado:** mejore iluminación, distancia y posición; evite lentes oscuros, mascarilla o rostro de perfil.
- **Varias personas:** el modelo puede devolver análisis por rostro; este notebook está diseñado principalmente para una captura con una persona.
- **Limitación importante:** las expresiones faciales no revelan de forma confiable el estado emocional interno. El resultado depende de iluminación, cultura, pose, calidad de imagen y sesgos del conjunto de entrenamiento.

### Preguntas para el ejercicio

1. ¿La predicción cambia con iluminación, distancia o ángulo?
2. ¿Qué ocurre si aparecen dos o más personas?
3. ¿Por qué una emoción estimada no equivale a conocer el estado emocional real?
4. ¿Qué diferencia observa entre detección facial y reconocimiento de identidad?
5. ¿Qué medidas de consentimiento y privacidad aplicaría antes de usar esta técnica con otras personas?